# The Song or the Machine? Dashboard launcher

This notebook starts the Streamlit dashboard (`app.py`) and shows it right here.
The dashboard only *reads* the files in `results/`; it does not compute anything itself.

## How to run it
1. **Open this notebook from the project folder**, the one that contains `app.py`, `dashboard/` and `results/`.
2. **Pick the right kernel:** the project's `.venv` (in VS Code / PyCharm / Jupyter: *Select Kernel → .venv*).
   No `.venv` yet? Run `uv sync` in a terminal inside the project folder first.
3. **Run the cells from top to bottom** (*Run All* works).
   - Cell 1 checks the packages are installed.
   - Cell 2 checks the result files exist.
   - Cell 3 starts the dashboard and prints a link.
   - Cell 4 shows it inside the notebook.
4. **Open the printed link** (usually http://localhost:8501) in your browser for the full-size version. That's the best view for presenting.
5. **When you're done, run the last cell** to stop the dashboard. Restarting or closing the kernel also stops it.

**Updating the numbers:** re-run `Jakob_i2pdp.ipynb`, `idil_I2pdp_final(2).ipynb` and `export_data_overview.py`. They rewrite `results/`. Then refresh the dashboard page in your browser.

## 1. Check the packages

In [3]:
import importlib.util, subprocess, sys

missing = [p for p in ("streamlit", "pandas", "plotly") if importlib.util.find_spec(p) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All packages installed. Python:", sys.executable)

All packages installed. Python: /Users/can/Documents/masters/I2DPD/project/.venv/bin/python


## 2. Check the result files
The dashboard needs these files in `results/`. If one is missing, re-run the notebook or script named next to it.

In [5]:
from pathlib import Path

PROJECT = Path.cwd()
assert (PROJECT / "app.py").exists(), "app.py not found: open this notebook from the project folder."

SOURCES = {
    "stats_": "Jakob_i2pdp.ipynb",
    "ml_": "idil_I2pdp_final(2).ipynb",
    "data_": "export_data_overview.py",
}
for prefix, source in SOURCES.items():
    files = sorted(p.name for p in (PROJECT / "results").glob(prefix + "*"))
    status = f"{len(files)} files" if files else "MISSING: run " + source
    print(f"{prefix:7s} -> {status}")
print("labels.csv ->", "ok" if (PROJECT / "results" / "labels.csv").exists() else "MISSING")

stats_  -> 7 files
ml_     -> 9 files
data_   -> 7 files
labels.csv -> ok


## 3. Start the dashboard
Starts Streamlit in the background and waits until it responds. If port 8501 is busy, it takes the next free one.

In [6]:
import socket, time, urllib.request

def free_port(start=8501):
    port = start
    while True:
        with socket.socket() as s:
            if s.connect_ex(("localhost", port)) != 0:
                return port
        port += 1

PORT = free_port()
URL = f"http://localhost:{PORT}"

dashboard = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(60):  # wait up to 30 seconds
    try:
        urllib.request.urlopen(URL + "/_stcore/health", timeout=1)
        print("Dashboard running at", URL)
        break
    except Exception:
        if dashboard.poll() is not None:
            print("Streamlit stopped with an error:\n", dashboard.stdout.read())
            break
        time.sleep(0.5)
else:
    print("Dashboard did not respond within 30 s. Check the output of the last cell.")

Dashboard running at http://localhost:8502


## 4. Show it inside the notebook
For presenting, the browser link above is better (full screen, no scrolling inside a frame).

In [7]:
from IPython.display import IFrame
IFrame(URL, width="100%", height=900)

## 5. Stop the dashboard

In [8]:
if dashboard.poll() is None:
    dashboard.terminate()
    dashboard.wait(timeout=10)
print("Dashboard stopped.")

Dashboard stopped.
